# Inference pipeline

At this point, you should have the following files for the models trained on data from BWC batches 160 to 200:

1. `models/stage1_best_model.joblib`
2. `models/stage2_best_model.joblib`

You should also have unseen gradesheet data (in .xlsx) that you want to run inference on in `data/unseen` folder.

To run evaluation against known ground truth labels, ground truth data (in .xlsx) must be in `data/labels` folder. 

This notebook is currently running inference on data from BWC batches 201 and onwards. Predictions will be saved to `predictions` folder.

## Imports, constants

In [1]:
import pandas as pd
import joblib
from pathlib import Path

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

MODULE_1 = [
    "GH 1",
    "GH 2",
    "GH 3",
    "GH 4",
    "GH 5",
    "GH 6",
    "GH 7",
    "GH 8",
    "GH 9",
    "GH 10",
    "GH 11",
    "GH 12",
    "GH 13",
    "GH 14",
    "GH 15",
    "GH 17",
    "GH 19",
    "GH 21",
    "GH 22",
]

MODULE_2 = [
    "GH 24",
    "GH 25",
    "GH 27",
    "GH 28",
    "GH 29",
    "GH 30",
    "GH 31",
    "GH 32",
    "GH 33",
    "GH 34",
    "GHT",
]

MODULE_3 = ["IF 1", "IF 2", "IF 3", "IF 4", "IF 5", "IF 6", "IFT"]

MODULE_6 = ["AN 1", "AN 2", "AN 3", "AN 4", "AN 5", "AN 6", "AN 7"]

LESSONS_TO_KEEP = MODULE_1 + MODULE_2 + MODULE_3

## Prepare data for inference

### Read raw unseen data

In [2]:
# Read raw unseen data
dfs = []
for file in Path("data/unseen").glob("*.xlsx"):
    df = pd.read_excel(file)
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 163191 entries, 0 to 163190
Data columns (total 73 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   STUDENT_ID                      163191 non-null  str           
 1   STUDENT_FIRST_NAME              163191 non-null  str           
 2   STUDENT_LAST_NAME               163191 non-null  str           
 3   STUDENT_DISPLAY_NAME            163191 non-null  str           
 4   STUDENT_TITLE                   163191 non-null  str           
 5   STUDENT_RANK                    0 non-null       float64       
 6   STUDENT_RANK_ABBREV             0 non-null       float64       
 7   COURSE                          103154 non-null  str           
 8   CLASS_TEMP_DS_ID                101101 non-null  str           
 9   CLASS                           163191 non-null  str           
 10  CLASS_DS_ID                     101101 non-null  str           
 11

### Clean data

In [3]:
# Drop duplicate rows
df = df.drop_duplicates()

# Drop rows with null values in key columns
df = df.dropna(subset=["STUDENT_ID", "LESSON_NUMBER", "EMP_EVAL_SCORE"])

# Keep only the relevant lesson numbers (modules 1 to 3)
# This step also removes rows with suffixes.
df = df[df["LESSON_NUMBER"].isin(LESSONS_TO_KEEP)]

# Drop rows with NG in EMP_EVAL_COMMENTS
df = df[
    ~df["EMP_EVAL_COMMENTS"]
    .astype("string")
    .str.contains(r"\bNG\b", case=False, na=False)
]

# Drop rows with DNCO
df = df[
    ~(
        df["DUTY_STATUS_DESCRIPTION"].astype("string").eq("DNCO")
        | df["EMP_EVAL_COMMENTS"]
        .astype("string")
        .str.contains(r"\bDNCO\b", case=False, na=False)
    )
]

# Keep only the first row for a given STUDENT_ID and LESSON_NUMBER
df = df.drop_duplicates(subset=["STUDENT_ID", "LESSON_NUMBER", "LESSON_ACTUAL_START_DATE"])

# Reorder columns for readability
first_cols = [
    "STUDENT_ID",
    "LESSON_NUMBER",
    "LESSON_ACTUAL_START_DATE",
    "LESSON_ACTUAL_END_DATE",
    "EMP_EVAL_SCORE",
    "EMP_EVAL_COMMENTS",
    "DUTY_STATUS_DESCRIPTION",
    "ADJUSTED_SCORE",
    "AVERAGE_SCORE",
    "OBJECTIVE_DESCRIPTION",
    "OBJECTIVE_RAW_SCORE",
    "OBJECTIVE_SCORE_WEIGHT",
    "OBJECTIVE_WEIGHTED_SCORE",
    "OBJECTIVE_CRITICAL",
    "OBJECTIVE_ACCEPTABLE_SCORE",
    "OBJECTIVE_ACCEPTABLE_SCORE_DSC",
]
remaining_cols = [col for col in df.columns if col not in first_cols]
df = df[first_cols + remaining_cols]

# Sort by STUDENT_ID
df = df.sort_values(by=["STUDENT_ID", "LESSON_ACTUAL_START_DATE"]).reset_index(drop=True)

### Pivot

In [4]:
# Pivot
wide = (
    df.pivot_table(
        index="STUDENT_ID",
        columns="LESSON_NUMBER",
        values="EMP_EVAL_SCORE",
        aggfunc="first",
    )
    .reset_index()
)

# Reorder columns
wide = wide[["STUDENT_ID"] + LESSONS_TO_KEEP]
wide.columns.name = None
wide

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,GH 33,GH 34,GHT,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT
0,201EEZ,5.179442,5.267281,5.234087,4.355000,4.045916,4.367075,4.779347,3.446281,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,201HLEE,4.624989,4.917051,4.332565,4.444498,4.816778,3.785408,4.155844,4.118668,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,201HONT,5.113508,4.917051,4.133894,4.671514,4.773037,4.735986,4.372067,3.616723,NaN,...,NaN,4.101320,4.173327,5.027020,4.245510,4.539404,4.406282,3.777039,4.195178,4.557287
3,201KHOOC,5.301836,4.977205,4.503209,4.634096,4.935050,4.214286,4.815559,4.484804,NaN,...,NaN,4.357262,4.421186,5.377912,5.277256,4.344802,4.386612,NaN,4.888308,4.204591
4,201KOHJ,5.113508,4.917051,4.266631,4.425050,4.351428,4.299694,3.487584,3.593985,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,208TEOC,5.610000,5.840000,4.710000,5.210000,4.900000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,208TOHZ,5.230000,5.130000,3.920000,4.260000,4.180000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
112,208TONGZ,5.900000,5.990000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,208WONGW,5.520000,5.470000,4.800000,4.540000,4.410000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Mod_3 Completion Status

In [5]:
# Module 3 completion status (1 = complete, 0 = incomplete)
wide["Mod_3_completion_status"] = (
    wide[MODULE_3].notna().all(axis=1).astype(int)
)

## Run inference

In [6]:
# Load trained model
stage1_best_model = joblib.load("models/stage1_best_model.joblib")
stage2_best_model = joblib.load("models/stage2_best_model.joblib")

### Stage 1: Predict BWC pass or fail

In [7]:
# Keep only the feature columns
# X1 = wide.drop(columns="STUDENT_ID")
X1 = wide.drop(columns=["STUDENT_ID", "Mod_3_completion_status"])

# Get predictions and associated probability
y_pred1 = stage1_best_model.predict(X1)
y_score1 = stage1_best_model.predict_proba(X1)[:, 1]

# Append back to wide span table
wide["pred_bwc"] = y_pred1
wide["prob_bwc"] = y_score1
assert wide["pred_bwc"].isnull().sum() == 0
wide

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,Mod_3_completion_status,pred_bwc,prob_bwc
0,201EEZ,5.179442,5.267281,5.234087,4.355000,4.045916,4.367075,4.779347,3.446281,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.118992
1,201HLEE,4.624989,4.917051,4.332565,4.444498,4.816778,3.785408,4.155844,4.118668,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.065594
2,201HONT,5.113508,4.917051,4.133894,4.671514,4.773037,4.735986,4.372067,3.616723,NaN,...,5.027020,4.245510,4.539404,4.406282,3.777039,4.195178,4.557287,1,1.0,0.881256
3,201KHOOC,5.301836,4.977205,4.503209,4.634096,4.935050,4.214286,4.815559,4.484804,NaN,...,5.377912,5.277256,4.344802,4.386612,NaN,4.888308,4.204591,0,1.0,0.902146
4,201KOHJ,5.113508,4.917051,4.266631,4.425050,4.351428,4.299694,3.487584,3.593985,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.014283
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,208TEOC,5.610000,5.840000,4.710000,5.210000,4.900000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.121800
111,208TOHZ,5.230000,5.130000,3.920000,4.260000,4.180000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.014030
112,208TONGZ,5.900000,5.990000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.132382
113,208WONGW,5.520000,5.470000,4.800000,4.540000,4.410000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.012625


### Stage 2: Among predicted BWC passes, predict fighter or not

In [8]:
# Keep only predicted passes
passes = wide[wide["pred_bwc"] == 1].drop(columns=["pred_bwc", "prob_bwc"])

# Keep only the feature columns
# X2 = passes.drop(columns=["STUDENT_ID"])
X2 = passes.drop(columns=["STUDENT_ID", "Mod_3_completion_status"])


# Get predictions and associated probability
y_pred2 = stage2_best_model.predict(X2)
y_score2 = stage2_best_model.predict_proba(X2)[:, 1]

# Append back to wide span table
passes["pred_fighter"] = y_pred2
passes["prob_fighter"] = y_score2
assert passes["pred_fighter"].isnull().sum() == 0
passes

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,Mod_3_completion_status,pred_fighter,prob_fighter
2,201HONT,5.113508,4.917051,4.133894,4.671514,4.773037,4.735986,4.372067,3.616723,NaN,...,5.027020,4.245510,4.539404,4.406282,3.777039,4.195178,4.557287,1,0.0,0.365022
3,201KHOOC,5.301836,4.977205,4.503209,4.634096,4.935050,4.214286,4.815559,4.484804,NaN,...,5.377912,5.277256,4.344802,4.386612,NaN,4.888308,4.204591,0,1.0,0.523451
5,201KOHZ,5.813411,6.180373,5.756446,4.930210,4.751955,4.814494,5.047512,5.502013,NaN,...,5.071061,4.491836,4.585104,3.962011,4.507744,3.785039,4.760613,1,1.0,0.699344
6,201KSARAN,5.301836,4.977205,4.398210,4.249666,4.473469,4.039001,4.233766,4.197280,NaN,...,4.270524,4.368571,4.306142,4.388513,3.631730,3.957875,4.258857,1,0.0,0.234078
7,201LEEJ,5.116952,4.977205,5.101195,5.016398,5.315733,4.388821,4.257563,4.376099,NaN,...,4.628361,4.860992,4.438608,4.468097,4.295469,3.913083,4.211606,1,0.0,0.467614
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,206TANS,5.361741,5.222026,4.722091,4.696520,4.453276,4.408988,4.409179,4.437363,NaN,...,4.569988,4.000000,4.229084,4.270312,4.051228,NaN,NaN,0,0.0,0.340403
82,206YEOG,4.994169,4.972374,4.711017,4.681913,4.614198,4.543613,4.544351,4.473858,NaN,...,4.285507,5.207348,4.461302,4.231225,3.945963,4.075004,4.246296,1,0.0,0.471387
83,206YEWP,5.909427,6.030530,5.409156,5.210907,4.412720,4.582246,4.814891,4.349187,NaN,...,4.539208,4.782785,4.461302,5.212592,4.161722,4.584353,4.360107,1,1.0,0.663507
92,207LIMJ,5.053561,5.036647,5.530314,5.169496,4.727366,4.519926,5.660709,5.403302,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1.0,0.851795


### Final output

In [9]:
predictions = pd.merge(
    wide[["STUDENT_ID", "pred_bwc", "prob_bwc", "Mod_3_completion_status"]],
    passes[["STUDENT_ID", "pred_fighter", "prob_fighter"]],
    on="STUDENT_ID",
    how="left",
)
predictions = predictions.fillna(0.0)
predictions = predictions[
    [
        "STUDENT_ID",
        "pred_bwc",
        "prob_bwc",
        "pred_fighter",
        "prob_fighter",
        "Mod_3_completion_status",
    ]
]

In [10]:
predictions.head(20)

,STUDENT_ID,pred_bwc,prob_bwc,pred_fighter,prob_fighter,Mod_3_completion_status
0,201EEZ,0.0,0.118992,0.0,0.000000,0
1,201HLEE,0.0,0.065594,0.0,0.000000,0
2,201HONT,1.0,0.881256,0.0,0.365022,1
3,201KHOOC,1.0,0.902146,1.0,0.523451,0
4,201KOHJ,0.0,0.014283,0.0,0.000000,0
5,201KOHZ,1.0,0.741961,1.0,0.699344,1
6,201KSARAN,1.0,0.938351,0.0,0.234078,1
7,201LEEJ,1.0,0.939199,0.0,0.467614,1
8,201LIMK,1.0,0.752536,0.0,0.470441,0
9,201ONGE,1.0,0.933237,1.0,0.536148,1


In [11]:
predictions.tail(20)

,STUDENT_ID,pred_bwc,prob_bwc,pred_fighter,prob_fighter,Mod_3_completion_status
95,207SEAHI,0.0,0.083302,0.0,0.0,0
96,207SOHY,0.0,0.475025,0.0,0.0,0
97,207TANK,0.0,0.137841,0.0,0.0,0
98,207TIONGT,0.0,0.100163,0.0,0.0,0
99,208BHAVESH,0.0,0.417485,0.0,0.0,0
100,208HOJ,0.0,0.148447,0.0,0.0,0
101,208JKOK,0.0,0.159129,0.0,0.0,0
102,208KOHS,0.0,0.025514,0.0,0.0,0
103,208KOHZ,0.0,0.010359,0.0,0.0,0
104,208LAIL,0.0,0.011417,0.0,0.0,0


In [12]:
predictions.to_csv("predictions/201-208_predictions.csv", index=False)

## Evaluate against known ground truth

Ground truth is taken from `data/labels/BWC201t207_Status_as_of_end_Feb2026.xlsx`, as sent by Tommy to Melody on Defence mail on 13 March 2026.

### Get ground truth

In [13]:
# Load ground truth
ground_truth = pd.read_excel("data/labels/BWC201t208_Status_as_of_end_Mar2026.xlsx")
ground_truth = ground_truth.rename(columns={"STUDENT_ID_ORIG": "STUDENT_ID"})

# Create target columns
ground_truth["target_bwc"] = pd.NA
ground_truth.loc[ground_truth["BWC_Status"] == "Fail", "target_bwc"] = 0.0
ground_truth.loc[
    ground_truth["BWC_Status"].isin(["Pass", "DNF(SUPT)"]), "target_bwc"
] = 1.0
ground_truth["target_bwc"].astype("Float64")

ground_truth["target_fighter"] = pd.NA
ground_truth.loc[
    ((ground_truth["target_bwc"].notna()) & (ground_truth["Stream_Group"] != "Fighter")),
    "target_fighter",
] = 0.0
ground_truth.loc[
    ((ground_truth["target_bwc"] == 1.0) & (ground_truth["Stream_Group"] == "Fighter")),
    "target_fighter",
] = 1.0
ground_truth["target_bwc"].astype("Float64")

if len(ground_truth) != len(predictions):
    print(set(ground_truth["STUDENT_ID"].unique()) - set(predictions["STUDENT_ID"].unique()))
    print(set(predictions["STUDENT_ID"].unique()) - set(ground_truth["STUDENT_ID"].unique()))

{'202LIMN'}
set()


In [14]:
final = pd.merge(ground_truth, predictions, on="STUDENT_ID", how="left")

# Drop rows where target_BWC is nan, i.e. Ongoing, DNF due to med, disciplinary, etc.
final = final.dropna(subset="target_bwc").reset_index(drop=True)

# Enforce datatype
for col in ["target_bwc", "target_fighter", "pred_bwc", "pred_fighter", "prob_bwc", "prob_fighter"]:
    final[col] = final[col].astype("Float64")

final = final.drop(columns=["Mod_3_completion_status"])    
final

,STUDENT_ID,Batch,BWC_Status,Stream_Group,Stream_Unit,target_bwc,target_fighter,pred_bwc,prob_bwc,pred_fighter,prob_fighter
0,201EEZ,201,Fail,NaN,NaN,0.0,0.0,0.0,0.118992,0.0,0.0
1,201HLEE,201,Fail,NaN,NaN,0.0,0.0,0.0,0.065594,0.0,0.0
2,201HONT,201,Fail,NaN,NaN,0.0,0.0,1.0,0.881256,0.0,0.365022
3,201KOHJ,201,Fail,NaN,NaN,0.0,0.0,0.0,0.014283,0.0,0.0
4,201LIMK,201,Fail,NaN,NaN,0.0,0.0,1.0,0.752536,0.0,0.470441
...,...,...,...,...,...,...,...,...,...,...,...
59,206MATHEWSL,206,Fail,NaN,NaN,0.0,0.0,0.0,0.111284,0.0,0.0
60,206TANA,206,Fail,NaN,NaN,0.0,0.0,0.0,0.034931,0.0,0.0
61,207ANGD,207,Fail,NaN,NaN,0.0,0.0,0.0,0.027212,0.0,0.0
62,207LIMH,207,Fail,NaN,NaN,0.0,0.0,0.0,0.021871,0.0,0.0


### Evaluation

In [15]:
final["pred_fighter"].value_counts(dropna=False)

pred_fighter
0.0    43
1.0    21
Name: count, dtype: Int64

In [16]:
print("Stage 1: Predict pass or fail BWC")
print(
    classification_report(
        final["target_bwc"].astype("Int64"),
        final["pred_bwc"].astype("Int64"),
        zero_division=0,
        digits=4,
    )
)
print(f"ROC-AUC: {roc_auc_score(final['target_bwc'], final['prob_bwc']):.4f}")
tn, fp, fn, tp = confusion_matrix(final["target_bwc"], final["pred_bwc"]).ravel()
print("Confusion Matrix:")
print(f"TN: {tn}, FP: {fp}")
print(f"FN: {fn}, TP: {tp}")

print()

print("Stage 2: Among predicted passes, predict streamed to fighter or not")
print(
    classification_report(
        final["target_fighter"].astype("Int64"),
        final["pred_fighter"].astype("Int64"),
        zero_division=0,
        digits=4,
    )
)
print(f"ROC-AUC: {roc_auc_score(final['target_fighter'], final['prob_fighter']):.4f}")
tn, fp, fn, tp = confusion_matrix(
    final["target_fighter"], final["pred_fighter"]
).ravel()
print("Confusion Matrix:")
print(f"TN: {tn}, FP: {fp}")
print(f"FN: {fn}, TP: {tp}")

Stage 1: Predict pass or fail BWC
              precision    recall  f1-score   support

         0.0     0.9500    0.8636    0.9048        22
         1.0     0.9318    0.9762    0.9535        42

    accuracy                         0.9375        64
   macro avg     0.9409    0.9199    0.9291        64
weighted avg     0.9381    0.9375    0.9367        64

ROC-AUC: 0.9567
Confusion Matrix:
TN: 19, FP: 3
FN: 1, TP: 41

Stage 2: Among predicted passes, predict streamed to fighter or not
              precision    recall  f1-score   support

         0.0     0.8372    0.9000    0.8675        40
         1.0     0.8095    0.7083    0.7556        24

    accuracy                         0.8281        64
   macro avg     0.8234    0.8042    0.8115        64
weighted avg     0.8268    0.8281    0.8255        64

ROC-AUC: 0.8984
Confusion Matrix:
TN: 36, FP: 4
FN: 7, TP: 17


In [17]:
final.to_csv("predictions/201-207_predictions_with_ground_truth.csv", index=False)